# 📉 Chapter 3: Dimensionality Reduction Techniques
**Referensi Buku:** *scikit-learn Cookbook, Third Edition*

---
## 1. Pendahuluan
Ketika kita memiliki dataset dengan jumlah fitur (kolom) yang sangat banyak, model bisa menjadi lambat dan rentan terhadap fenomena *Curse of Dimensionality*. Bab ini membahas berbagai resep untuk mengurangi dimensi data tanpa kehilangan terlalu banyak informasi penting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, make_circles

%matplotlib inline
np.random.seed(42)

iris = load_iris()
X = iris.data
y = iris.target
print(f"Bentuk data asli: {X.shape} (Memiliki 4 fitur/dimensi)")

## 2. Principal Component Analysis (PCA)
PCA adalah teknik paling standar untuk reduksi dimensi. Ia mencari kombinasi linier dari fitur-fitur asli yang dapat menangkap variansi (informasi) terbesar dari data. Kita akan mereduksi data Iris dari 4 dimensi menjadi 2 dimensi agar bisa divisualisasikan pada grafik.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# PCA sangat sensitif terhadap skala, jadi kita harus melakukan standarisasi dulu
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Bentuk data setelah PCA: {X_pca.shape}")
print(f"Rasio Variansi yang dipertahankan: {sum(pca.explained_variance_ratio_) * 100:.2f}%")

plt.figure(figsize=(7, 5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', edgecolor='k')
plt.title('PCA: Proyeksi Iris Dataset ke 2D')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(handles=scatter.legend_elements()[0], labels=list(iris.target_names))
plt.show()

## 3. Kernel PCA (Untuk Data Non-Linier)
PCA biasa hanya bisa menangkap hubungan linier. Jika data berbentuk melingkar atau kompleks (non-linier), PCA biasa akan gagal memisahkannya. Kita menggunakan **Kernel PCA** (dengan trik kernel seperti fungsi RBF) untuk membentangkan data non-linier ke dimensi yang lebih tinggi agar bisa dipisahkan.

In [ ]:
from sklearn.decomposition import KernelPCA

# Membuat data non-linier (Dua lingkaran yang saling mengelilingi)
X_circle, y_circle = make_circles(n_samples=400, factor=0.3, noise=0.05, random_state=42)

# Menggunakan Kernel PCA dengan kernel RBF
kpca = KernelPCA(kernel='rbf', gamma=10, n_components=2)
X_kpca = kpca.fit_transform(X_circle)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(X_circle[:, 0], X_circle[:, 1], c=y_circle, cmap='bwr', edgecolor='k')
ax[0].set_title('Data Asli (Bentuk Lingkaran)')

ax[1].scatter(X_kpca[:, 0], X_kpca[:, 1], c=y_circle, cmap='bwr', edgecolor='k')
ax[1].set_title('Setelah Kernel PCA (Data berhasil direntangkan)')
plt.show()

## 4. Truncated SVD (Singular Value Decomposition)
PCA standar memusatkan data (mengurangi nilainya dengan rata-rata/mean) sebelum bekerja. Ini merusak integritas matriks sparse (matriks yang sebagian besar isinya adalah nol), seperti representasi data teks TF-IDF. **Truncated SVD** adalah alternatif PCA yang dirancang khusus untuk bekerja langsung pada Sparse Matrix tanpa merusak efisiensi memorinya.

In [ ]:
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix

# Membuat matriks sparse (banyak nol)
X_sparse = csr_matrix([
    [1, 0, 0, 0, 2],
    [0, 0, 3, 0, 0],
    [0, 0, 0, 4, 0],
    [5, 0, 0, 0, 6]
])

svd = TruncatedSVD(n_components=2)
X_svd = svd.fit_transform(X_sparse)

print("Matriks Sparse Asli (Bentuk 4x5):")
print(X_sparse.toarray())
print("\nSetelah Truncated SVD (Bentuk 4x2):")
print(X_svd)

## 5. Factor Analysis
Berbeda dengan PCA yang sekadar merangkum variansi matriks secara matematis, Factor Analysis adalah model generatif. Ia mengasumsikan bahwa data yang kita lihat dihasilkan oleh beberapa 'Faktor Laten' (variabel tersembunyi yang tidak teramati) yang dicampur dengan *noise* (gangguan).

In [ ]:
from sklearn.decomposition import FactorAnalysis

fa = FactorAnalysis(n_components=2, random_state=42)
X_fa = fa.fit_transform(X_scaled)

plt.figure(figsize=(7, 5))
scatter_fa = plt.scatter(X_fa[:, 0], X_fa[:, 1], c=y, cmap='plasma', edgecolor='k')
plt.title('Factor Analysis pada Iris Dataset')
plt.xlabel('Latent Factor 1')
plt.ylabel('Latent Factor 2')
plt.legend(handles=scatter_fa.legend_elements()[0], labels=list(iris.target_names))
plt.show()